# Build the non-AI control group (offline, no API)

Turns a **downloaded CourtListener bulk file** into a coded control group of non-AI sanctions
cases, using only pandas and regex. No API, no token.

## Before you run
1. Download one file from CourtListener's public bulk bucket (free, no account):
   `opinion-clusters-YYYY-MM-DD.csv.bz2` (~2.3 GB compressed). Either click it on the bulk-data
   page, or:
   ```
   aws s3 cp s3://com-courtlistener-storage/bulk-data/opinion-clusters-2026-06-30.csv.bz2 ./bulk/ --no-sign-request
   ```
2. Put it in a `bulk/` folder next to this notebook. **Leave it compressed** — pandas reads `.bz2`
   directly, and decompressing just makes a ~12 GB file for no reason.
3. Keep `label_lib.py` and `analysis_data_coded.csv` next to this notebook.

## How it handles the size
The file has ~4M rows; the cases we want (sanctions, 2023–2026) are a few thousand. We never hold
the whole file in memory. We stream it in chunks, read only the ~7 columns we need, filter each
chunk down to sanctions cases, and keep only those. Peak memory is one chunk. The 2.3 GB passes
*through* the machine; almost nothing stays.

**Run order:** config → coders → preflight (fast sanity check) → full run (the long pass) → stack.

## 1. Config

In [ ]:
import re

# ---- input ----
SNAPSHOT_DATE = "2026-06-30"                                   # for provenance; cite this
BULK_CSV      = f"/Users/jud._.e/Documents/GitHub/qss20-ai_hallucinations_project/labeling/bulk-data/opinion-clusters-{SNAPSHOT_DATE}.csv.bz2"
AI_CODED      = "/Users/jud._.e/Documents/GitHub/qss20-ai_hallucinations_project/labeling/analysis_data_coded.csv"   # the AI arm you already have

# ---- filters ----
YEAR_MIN, YEAR_MAX = 2023, 2026
SANCTION_TRIGGER = re.compile(
    r"rule\s*11|§?\s*1927|section\s*1927|inherent\s+authority|sanction|"
    r"show\s+cause|disciplin|referr", re.I)

# ---- performance / sampling ----
CHUNKSIZE          = 100_000      # lower to 50_000 on an 8 GB machine
CONTROL_SAMPLE_CAP = 2000         # keep at most this many controls (plenty for the regression)
RANDOM_SEED        = 20

# ---- output ----
OUT_CONTROLS = "/Users/jud._.e/Documents/GitHub/qss20-ai_hallucinations_project/labeling/controls_coded.csv"
OUT_POOLED   = "/Users/jud._.e/Documents/GitHub/qss20-ai_hallucinations_project/labeling/pooled_coded.csv"

# ---- CourtListener opinion-clusters column names (edit if a future schema renames them) ----
COL = dict(
    case="case_name", date="date_filed", nos="nature_of_suit", attorneys="attorneys",
    text_cols=["disposition","summary","procedural_history","posture","syllabus","headnotes"])

## 2. Coders
These sit on top of `label_lib`. `field` comes from the Nature-of-Suit string; `pro_se` comes from
the `attorneys` field, falling back to pro-se markers in the text when that field is blank.

In [ ]:
import pandas as pd, numpy as np, os
import label_lib as L

def build_text(df):
    """Vectorized: concatenate the available text columns into one blob per row."""
    cols=[c for c in COL["text_cols"] if c in df.columns]
    s=pd.Series("", index=df.index, dtype="object")
    for c in cols:
        s=s.str.cat(df[c].fillna("").astype(str), sep="  ")
    return s

def field_from_nos_text(nos):
    """CourtListener NOS is usually 'CODE Label' (e.g. '190 Contract'). Try code, then keywords."""
    if not isinstance(nos,str) or not nos.strip(): return None
    m=re.match(r"\s*(\d{3})",nos)
    if m:
        f=L.field_from_nos(m.group(1))
        if f: return f
    s=nos.lower()
    for key,val in [("civil right","civil rights"),("contract","contract"),("tort","tort"),
                    ("employ","employment"),("labor","employment"),("administrativ","administrative"),
                    ("family","family"),("bankrupt","bankruptcy"),("immigrat","immigration"),
                    ("patent","IP"),("trademark","IP"),("copyright","IP"),("habeas","habeas"),("tax","tax")]:
        if key in s: return val
    return "other"

def pro_se_from_attorneys(att):
    """'pro se'/'pro per' -> 1; a non-empty attorney list -> 0; empty/unknown -> None."""
    if not isinstance(att,str) or not att.strip(): return None
    if re.search(r"pro\s+se|pro\s+per|self[-\s]represent", att, re.I): return 1
    return 0

def process_chunk(df, verbose=False):
    """One chunk in, a small coded frame out. Filters are ordered cheap-to-expensive:
    vectorized window + trigger first, then the per-row coders run on the survivors only."""
    df=df.rename(columns={COL["case"]:"case_name", COL["date"]:"date_filed",
                          COL["nos"]:"nature_of_suit", COL["attorneys"]:"attorneys"})
    n0=len(df)
    df["date"]=pd.to_datetime(df.get("date_filed"), errors="coerce")
    df["year"]=df["date"].dt.year
    df=df[df["year"].between(YEAR_MIN, YEAR_MAX)]                      # 1. window (vectorized)
    n1=len(df)
    df["_text"]=build_text(df)
    df=df[df["_text"].str.contains(SANCTION_TRIGGER, na=False)]        # 2. sanctions (vectorized)
    n2=len(df)
    df=df[~df["_text"].map(L.is_ai_contaminated)]                     # 3. drop hidden AI cases
    n3=len(df)
    if len(df):                                                       # 4. code the survivors
        att = df["attorneys"] if "attorneys" in df.columns else pd.Series(index=df.index, dtype="object")
        nos = df["nature_of_suit"] if "nature_of_suit" in df.columns else pd.Series(index=df.index, dtype="object")
        df["severity"]=df["_text"].map(lambda t: L.code_severity(t, is_full_opinion=True))
        df["pro_se"]=[pro_se_from_attorneys(a) if pro_se_from_attorneys(a) is not None
                      else L.code_pro_se(text=t) for a,t in zip(att, df["_text"])]
        df["field"]=nos.map(field_from_nos_text)
    df["ai"]=0; df["federal"]=np.nan
    df["text_excerpt"]=df["_text"].str.slice(0,300)
    if verbose:
        print(f"    {n0:>7,} rows -> window {n1:>6,} -> sanctions {n2:>4,} -> non-AI {n3:>4,}")
    keep=["case_name","date_filed","year","ai","severity","pro_se","field","federal","nature_of_suit","text_excerpt"]
    return df.reindex(columns=keep)

## 3. Preflight (do this first)
Reads only the first chunk. Confirms the file opens, the column names match `COL`, and the filters
actually catch some sanctions cases. If a name is wrong you find out in seconds, not 20 minutes.

In [ ]:
assert os.path.exists(BULK_CSV), f"Not found: {BULK_CSV}. Download it first (see top)."
keep=set([COL["case"],COL["date"],COL["nos"],COL["attorneys"]]+COL["text_cols"])
first=next(pd.read_csv(BULK_CSV, usecols=lambda c: c in keep, dtype=str,
                       on_bad_lines="skip", chunksize=CHUNKSIZE))
print("columns present in file:", list(first.columns))
missing=[n for n in [COL["case"],COL["date"],COL["nos"],COL["attorneys"]] if n not in first.columns]
print("REQUIRED columns missing (fix COL if any):", missing or "none")
print("text columns found:", [c for c in COL["text_cols"] if c in first.columns])
print("\nfilter funnel on first chunk:")
demo=process_chunk(first, verbose=True)
print("\nsample of coded controls from first chunk:")
print(demo.dropna(subset=["severity"]).head(5)[["case_name","year","severity","pro_se","field"]].to_string(index=False))

## 4. Full run (the long pass)
Streams the whole file, keeps only sanctions cases. Expect roughly 10–30 minutes; the slow part is
`.bz2` decompression, not the filtering. `on_bad_lines="skip"` means one malformed row can't kill
the run. Peak memory stays around one chunk.

In [ ]:
reader=pd.read_csv(BULK_CSV, usecols=lambda c: c in keep, dtype=str,
                   on_bad_lines="skip", chunksize=CHUNKSIZE)
parts=[]; scanned=0
for i,ch in enumerate(reader):
    parts.append(process_chunk(ch)); scanned+=len(ch)
    if (i+1)%10==0:
        kept=sum(len(p) for p in parts)
        print(f"  scanned {scanned:>10,} rows | kept {kept:>5,} sanctions cases")
controls=pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
controls=controls.dropna(subset=["severity"]).drop_duplicates(subset=["case_name","date_filed"])
print(f"\ntotal sanctions controls found: {len(controls):,}")

if len(controls) > CONTROL_SAMPLE_CAP:                    # sample down; we need a comparison set, not all of it
    controls=controls.sample(CONTROL_SAMPLE_CAP, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"randomly sampled down to {len(controls):,} (seed {RANDOM_SEED})")

controls.attrs["snapshot"]=SNAPSHOT_DATE
controls.to_csv(OUT_CONTROLS, index=False)
print(f"wrote {OUT_CONTROLS}  (CourtListener snapshot {SNAPSHOT_DATE})")
print("severity dist:", controls["severity"].value_counts().sort_index().to_dict())
print("pro_se coded (non-null):", int(controls["pro_se"].notna().sum()), "/", len(controls))

## 5. Stack under the AI arm
Builds `pooled_coded.csv` with an `ai` dummy (1 = Charlotin AI case, 0 = CourtListener control).
This is the file the pooled severity regression reads: `severity ~ ai + field + pro_se + year`,
where the `ai` coefficient is the reviewer's AI-vs-non-AI question.

In [ ]:
if os.path.exists(OUT_CONTROLS) and os.path.exists(AI_CODED):
    ai=pd.read_csv(AI_CODED)
    ai_slim=pd.DataFrame({
        "case_name":ai.get("Case Name"), "date_filed":ai.get("Date"), "year":ai.get("year"),
        "ai":1, "severity":ai.get("severity"), "pro_se":ai.get("pro_se"),
        "field":ai.get("field"), "federal":ai.get("federal"),
        "nature_of_suit":ai.get("Legal Field Primary"), "text_excerpt":ai.get("Outcome")})
    controls=pd.read_csv(OUT_CONTROLS)
    pooled=pd.concat([ai_slim, controls], ignore_index=True)
    pooled.to_csv(OUT_POOLED, index=False)
    print(f"wrote {OUT_POOLED}: {len(pooled):,} rows "
          f"(AI={int((pooled.ai==1).sum()):,}, control={int((pooled.ai==0).sum()):,})")
    print("\nmean severity by arm (raw, uncontrolled):")
    print(pooled.groupby("ai")["severity"].agg(["size","mean"]).round(2).to_string())
    print("\nNext: regress severity on ai + field + pro_se + year. Report the ai coefficient.")
    print("Caveats to keep: control frame is sanctions-adjacent non-AI cases; coding needs the")
    print("hand-validation from validate_coding.py; CourtListener skews federal/appellate.")
else:
    print("Need both", OUT_CONTROLS, "and", AI_CODED, "present. Run the full-run cell, and put")
    print("analysis_data_coded.csv next to this notebook.")